# Quick experiment

### Scope
**Graph representation** one node per word vs repeat as their appear
Test GCN performance on this.

**Model**: GCN

# Prepare dataset

In [1]:
import tqdm
import torch
import glob


# Unique node dataset.
UNIQUE_NODE_GRAPH_PATH = "../data/datasets/IMDB/@SET/interim/"

sets = ["train", "validation", "test"]

train_x_unique, val_x_unique, test_x_unique = [], [], []
train_y_unique, val_y_unique, test_y_unique = [], [], []

x_unique = {set_name: [] for set_name in sets}
y_unique = {set_name: [] for set_name in sets}

for set_name in sets:
    dataset_path = UNIQUE_NODE_GRAPH_PATH.replace("@SET", set_name)
    unique_node_graph_paths = glob.glob(dataset_path + "*.pt")

    for un_graph_path in tqdm.tqdm(unique_node_graph_paths,
                                   desc=f"Loading {set_name} set IMDB"):
        doc_name, label, graph_nx = torch.load(un_graph_path, weights_only=False)

        x_unique[set_name].append(graph_nx)
        y_unique[set_name].append(label)

Loading test set IMDB: 100%|██████████| 100/100 [00:01<00:00, 74.59it/s]


In [2]:
from pathlib import Path
import spacy
import networkx as nx

from src.data.preprocessing import preprocessing_imdb

# Loading the not unique.

nlp = spacy.load("en_core_web_lg")
deps = nlp.get_pipe("parser").labels
dep_label_map = {label: idx for idx, label in enumerate(deps)}

def build_structural_graph(target):
    doc = nlp(target)

    G = nx.MultiDiGraph()
    ent_map = {}

    # Collapse named entities into single nodes
    for ent in doc.ents:
        key = ent.text
        for i in range(ent.start, ent.end):
            ent_map[i] = key
        G.add_node(key, text=key, type="entity")

    # Add word nodes
    for token in doc:
        if token.i not in ent_map:
            G.add_node(token.text, text=token.text, type="word", position_in_sentence=token.i)

    # Add dependency edges with features
    for token in doc:
        head = token.head
        if token.i == head.i:
            continue
        src = ent_map.get(token.i, token.text)
        tgt = ent_map.get(head.i, head.text)
        feature = torch.zeros(len(dep_label_map))
        dep_idx = dep_label_map[token.dep_]
        feature[dep_idx] = 1.0
        G.add_edge(src, tgt, type="dep", feature=feature)

    return G

#####

IMDB_RAW_PATH = "../data/datasets/IMDB/@SET/raw/"

x_distinct = {set_name: [] for set_name in sets}
y_distinct = {set_name: [] for set_name in sets}

for set_name in sets:
    dataset_path = Path(IMDB_RAW_PATH.replace("@SET", set_name))
    for subfolder in dataset_path.iterdir():

        label = subfolder.name

        txt_files = list(subfolder.glob("*.txt"))[:5000]
        for txt_file in tqdm.tqdm(txt_files, desc=f"Loading set={set_name}, Label={label}"):
            with open(txt_file, "r") as f:
                text = f.read().strip()
            text = preprocessing_imdb(text, nlp)
            nx_graph = build_structural_graph(text)

            x_distinct[set_name].append(nx_graph)
            y_distinct[set_name].append(label)





/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
Loading set=train, Label=negative:  29%|██▉       | 1445/5000 [00:37<01:12, 48.95it/s]/media/trdp/STORAGE/3.Trabalho/PhD-Research/ThesisExperiments/3.ToValidate/TextualHGNN/src/data/preprocessing.py:436: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text_content = BeautifulSoup(raw_text, "html.parser").get_text()
Loading set=test, Label=positive: 100%|██████████| 2500/2500 [01:07<00:00, 37.02it/s]


# Training GCN e GAT

In [3]:
x_distinct.keys(), x_unique.keys()

(dict_keys(['train', 'validation', 'test']),
 dict_keys(['train', 'validation', 'test']))

In [4]:
x_distinct["train"][0], x_unique["train"][0], y_distinct["train"][0], y_unique["train"][0]
"train", "validation", "test"

('train', 'validation', 'test')

In [5]:
y_label_map = {label:i for i, label in enumerate(sorted(set(y_unique["train"])))}
num_classes = len(y_label_map)

y_label_map, num_classes

({'negative': 0, 'positive': 1}, 2)

In [6]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import networkx as nx
from typing import Dict, List, Literal

In [7]:
import networkx as nx
import torch
from torch_geometric.data import Data

# Define the label mapping
label_map = {'negative': 0, 'positive': 1}
num_classes = len(label_map)

def networkx_to_pyg(G: nx.MultiDiGraph, label: str) -> Data:
    """
    Converts a NetworkX MultiDiGraph to a PyTorch Geometric Data object.

    Implements our First Principles decisions:
    1.  Graph Structure: Converts to simple, undirected graph.
    2.  Node Features: Uses node degree (in + out) as a 1D feature.
    3.  Labels: Maps string label to an integer.
    """

    # --- THIS IS THE CORRECTED BLOCK ---
    # 1. Convert to undirected Graph (this collapses parallel edges)
    simple_g = nx.Graph(G.to_undirected())

    # 2. Remove self-loops
    simple_g.remove_edges_from(nx.selfloop_edges(simple_g))
    # --- END CORRECTION ---

    # 3. Create node mapping to contiguous integers [0, N-1]
    node_list = list(simple_g.nodes())
    node_map = {node: i for i, node in enumerate(node_list)}

    # 4. Create edge_index (COO format)
    edge_index = torch.tensor(
        [[node_map[u], node_map[v]] for u, v in simple_g.edges()],
        dtype=torch.long
    ).t().contiguous()

    # 5. Create node features (x)
    # **CRITICAL ASSUMPTION**: Using node degree as features.
    # If you have real features, replace this logic.
    features = []
    # TODO: Change to word embeddings
    for node in node_list:
        # Use degree from the original MultiDiGraph G
        degree = G.in_degree(node) + G.out_degree(node)
        features.append([float(degree)])

    x = torch.tensor(features, dtype=torch.float)

    # 6. Create label (y)
    y = torch.tensor([label_map[label]], dtype=torch.long)

    # Handle graphs with no nodes or no edges after simplification
    if x.shape[0] == 0:
        # If no nodes, create a dummy node feature tensor
        x = torch.zeros((0, 1), dtype=torch.float) # 0 nodes, 1 feature
        edge_index = torch.zeros((2, 0), dtype=torch.long) # 0 edges
    elif edge_index.shape[1] == 0:
        # If nodes but no edges, ensure edge_index is correctly shaped
        edge_index = torch.zeros((2, 0), dtype=torch.long) # 0 edges

    return Data(x=x, edge_index=edge_index, y=y)

In [8]:
print("Processing datasets...")
# Process all datasets
datasets = {}
for (name, x_data, y_data) in [('distinct', x_distinct, y_distinct), ('unique', x_unique, y_unique)]:
    datasets[name] = {}
    for split in ['train', 'validation', 'test']:
        print(f"Processing Dataset={name}, Split={split}")
        datasets[name][split] = [
            networkx_to_pyg(g, y) for g, y in zip(x_data[split], y_data[split])
        ]
    print(f"Finished processing '{name}' dataset.")
    print(f"  Sample 'train' data point: {datasets[name]['train'][0]}")

Processing datasets...
Processing Dataset=distinct, Split=train
Processing Dataset=distinct, Split=validation
Processing Dataset=distinct, Split=test
Finished processing 'distinct' dataset.
  Sample 'train' data point: Data(x=[102, 1], edge_index=[2, 151], y=[1])
Processing Dataset=unique, Split=train
Processing Dataset=unique, Split=validation
Processing Dataset=unique, Split=test
Finished processing 'unique' dataset.
  Sample 'train' data point: Data(x=[136, 1], edge_index=[2, 184], y=[1])


In [9]:
# === Step 2: Create PyG DataLoaders ===

BATCH_SIZE = 4

loaders = {}
for name in ['distinct', 'unique']:
    loaders[name] = {
        'train': DataLoader(datasets[name]['train'], batch_size=BATCH_SIZE, shuffle=True),
        'validation': DataLoader(datasets[name]['validation'], batch_size=BATCH_SIZE, shuffle=False),
        'test': DataLoader(datasets[name]['test'], batch_size=BATCH_SIZE, shuffle=False),
    }

# === Step 3: Define GNN Models ===

# Determine input feature dimension from the data
# (All graphs should have the same feature dim)
sample_data = datasets['distinct']['train'][0]
num_node_features = sample_data.num_node_features

In [10]:
class GraphClassifier(torch.nn.Module):
    def __init__(self, model_type: Literal['gcn', 'gat'], num_features: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.model_type = model_type
        self.hidden_dim = hidden_dim

        # GNN Backbone
        if model_type == 'gcn':
            self.conv1 = GCNConv(num_features, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, hidden_dim)
        elif model_type == 'gat':
            # Using 2 heads for the first layer
            self.conv1 = GATConv(num_features, hidden_dim // 2, heads=2, dropout=0.2)
            # Second layer GATConv (input dim is hidden_dim * heads)
            self.conv2 = GATConv(hidden_dim, hidden_dim, heads=1, concat=True, dropout=0.2)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        # Classifier Head
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        # 1. GNN layers
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # 2. Global Pooling (aggregates node features into graph feature)
        # `batch` tensor is provided by the DataLoader
        x_graph = global_mean_pool(x, batch)

        # 3. Classifier
        out = self.classifier(x_graph)
        return out

In [11]:
# === Step 4: Define Training & Evaluation Loops ===

def train_step(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def eval_step(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            loss = criterion(out, data.y)
            total_loss += loss.item() * data.num_graphs

            pred = out.argmax(dim=1)
            correct += int((pred == data.y).sum())

            # Collect preds and labels for classification report
            all_preds.append(pred)
            all_labels.append(data.y)

    # Concatenate all batch tensors
    final_preds = torch.cat(all_preds, dim=0).cpu()
    final_labels = torch.cat(all_labels, dim=0).cpu()

    accuracy = correct / len(loader.dataset)
    avg_loss = total_loss / len(loader.dataset)

    # Return all metrics
    return avg_loss, accuracy, final_labels, final_preds

In [12]:
from sklearn.metrics import classification_report

# === Step 5: Experiment Orchestration ===

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")
target_names = list(label_map.keys())

def run_experiment(dataset_name: str, model_type: Literal['gcn', 'gat'],
                   loaders: Dict, num_features: int, num_classes: int,
                   epochs: int = 20, hidden_dim: int = 64, lr: float = 0.001):

    print(f"--- Starting Experiment: [{dataset_name.upper()}] with [{model_type.upper()}] ---")

    # 1. Init Model, Optimizer, Criterion
    model = GraphClassifier(
        model_type=model_type,
        num_features=num_features,
        hidden_dim=hidden_dim,
        num_classes=num_classes
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    # Get the specific loaders for this experiment
    train_loader = loaders[dataset_name]['train']
    val_loader = loaders[dataset_name]['validation']
    test_loader = loaders[dataset_name]['test']

    # 2. Training Loop
    best_val_acc = 0
    for epoch in range(1, epochs + 1):
        train_loss = train_step(model, train_loader, optimizer, criterion, device)
        # We only need loss and acc for validation
        val_loss, val_acc, _, _ = eval_step(model, val_loader, criterion, device)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch: {epoch:02d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    # 3. Final Test Evaluation (capturing all return values)
    test_loss, test_acc, test_labels, test_preds = eval_step(model, test_loader, criterion, device)

    print(f"--- Finished Experiment ---")
    print(f"Final Test Accuracy: {test_acc:.4f}\n")

    # --- NEW: Generate and print classification report ---
    print("Test Set Classification Report:")
    try:
        report = classification_report(test_labels, test_preds, target_names=target_names)
        print(report)
    except Exception as e:
        print(f"Could not generate classification report: {e}")
        # This can happen if one class was never predicted, for example.
    print("\n" + "="*40 + "\n")
    # --- END NEW BLOCK ---

    return {
        "dataset": dataset_name,
        "model": model_type,
        "test_accuracy": test_acc
    }


Using device: cuda


In [ ]:
# === Step 6: Compare Results ===

# Define experiment configurations
experiments = [
    {'dataset_name': 'distinct', 'model_type': 'gcn'},
    {'dataset_name': 'distinct', 'model_type': 'gat'},
    {'dataset_name': 'unique', 'model_type': 'gcn'},
    {'dataset_name': 'unique', 'model_type': 'gat'},
]

results = []
for config in experiments:
    result = run_experiment(
        dataset_name=config['dataset_name'],
        model_type=config['model_type'],
        loaders=loaders,
        num_features=num_node_features,
        num_classes=num_classes,
        epochs=30 # Using 30 from before
    )
    results.append(result)

# Print final comparison table
print("="*40)
print("       FINAL EXPERIMENT RESULTS (Summary)       ")
print("="*40)
print(f"| {'Dataset':<10} | {'Model':<6} | {'Test Accuracy':<15} |")
print(f"|{'-'*12}|{'-'*8}|{'-'*17}|")
for res in results:
    print(f"| {res['dataset']:<10} | {res['model'].upper():<6} | {res['test_accuracy'] * 100:>14.2f}% |")
print("="*40)

--- Starting Experiment: [DISTINCT] with [GCN] ---
Epoch: 01, Train Loss: 0.6945, Val Loss: 0.6936, Val Acc: 0.5000
Epoch: 05, Train Loss: 0.6937, Val Loss: 0.6932, Val Acc: 0.5000
Epoch: 10, Train Loss: 0.6933, Val Loss: 0.6935, Val Acc: 0.5000
Epoch: 15, Train Loss: 0.6934, Val Loss: 0.6932, Val Acc: 0.5000
Epoch: 20, Train Loss: 0.6929, Val Loss: 0.6941, Val Acc: 0.5000
Epoch: 25, Train Loss: 0.6932, Val Loss: 0.6936, Val Acc: 0.5000
Epoch: 30, Train Loss: 0.6931, Val Loss: 0.6942, Val Acc: 0.5000
--- Finished Experiment ---
Final Test Accuracy: 0.5000

Test Set Classification Report:
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67      2500
    positive       0.00      0.00      0.00      2500

    accuracy                           0.50      5000
   macro avg       0.25      0.50      0.33      5000
weighted avg       0.25      0.50      0.33      5000



--- Starting Experiment: [DISTINCT] with [GAT] ---


/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

Epoch: 01, Train Loss: 0.6945, Val Loss: 0.6929, Val Acc: 0.5274
Epoch: 05, Train Loss: 0.6923, Val Loss: 0.6939, Val Acc: 0.5002
Epoch: 10, Train Loss: 0.6923, Val Loss: 0.6921, Val Acc: 0.5266
